# MSM

In [6]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

MSM은 일반적인 회귀분석이 실패하는 시간 의존적 교란(Time-dependent confounding) 상황에서 잘 작동하는 모델입니다.
- L (Covariate): 환자의 건강 상태 (예: CD4 수치). 이전 치료($A_{t-1}$)에 영향을 받습니다.
- A (Treatment): 치료 여부. 현재 건강 상태($L_t$)가 나쁠수록 치료받을 확률이 높아집니다.
- Feedback Loop: 치료는 건강을 좋게 만들지만, 건강이 나빠야 치료를 받습니다. 이 순환 고리 때문에 일반적인 조정(Adjustment)은 편향을 낳습니다.

In [ ]:


def generate_simulation_data(n=1000, t_points=3):
    """
    시간 의존적 교란(Time-dependent confounding)이 있는 데이터를 생성합니다.
    L: 교란 변수 (예: CD4 count)
    A: 치료 여부 (예: AZT)
    Y: 결과 (예: Viral load detectable)
    """
    np.random.seed(42)
    data = []

    for i in range(n):
        l_prev = 0  # 초기 L
        a_prev = 0  # 초기 A
        
        # 각 시점(t)별 데이터 생성
        for t in range(t_points):
            # L(t)는 이전 상태와 이전 치료에 영향을 받음 (Time-dependent confounder)
            # 치료(A)를 받으면 건강(L)이 좋아진다고 가정
            l_prob = 0.5 + 0.2 * l_prev - 0.1 * a_prev
            l_t = np.random.binomial(1, np.clip(l_prob, 0, 1))
            
            # A(t)는 현재의 L과 이전 치료 A에 영향을 받음 (Confounding by indication)
            # 상태(L)가 나쁘면 치료(A)를 받을 확률이 높다고 가정
            a_prob = 0.4 + 0.3 * l_t + 0.1 * a_prev
            a_t = np.random.binomial(1, np.clip(a_prob, 0, 1))
            
            data.append({
                'id': i,
                'time': t,
                'L': l_t,
                'A': a_t,
                'L_prev': l_prev,
                'A_prev': a_prev
            })
            
            l_prev = l_t
            a_prev = a_t

    df = pd.DataFrame(data)
    
    # 결과 변수 Y 생성 (마지막 시점에만 관측된다고 가정하거나, 누적 효과를 봄)
    # 실제 인과 효과: 치료(A)를 많이 받을수록 Y 발생 확률이 낮아짐 (beta_1 < 0)
    # 하지만 단순 관찰 데이터는 아픈 사람이 약을 먹으므로 연관성이 양수로 나올 수 있음
    total_dose = df.groupby('id')['A'].sum()
    last_l = df.groupby('id')['L'].last()
    
    y_prob = 0.6 - 0.15 * total_dose + 0.1 * last_l
    y_outcome = np.random.binomial(1, np.clip(y_prob, 0, 1))
    
    # 결과 변수 병합
    y_df = pd.DataFrame({'id': range(n), 'Y': y_outcome})
    df = df.merge(y_df, on='id')
    
    return df

# 데이터 로드
df = generate_simulation_data(n=2000, t_points=5)
print("Data Sample (Person-Time Format):")
print(df.head())


Data Sample (Person-Time Format):
   id  time  L  A  L_prev  A_prev  Y
0   0     0  0  1       0       0  0
1   0     1  1  1       0       1  0
2   0     2  1  1       1       1  0
3   0     3  1  0       1       1  0
4   0     4  1  0       1       0  0


## 1. IPTW 가중치 계산 (Calculating Stabilized Weights)
첫 단계는 가중치($SW$)를 계산하여 가상의 모집단(Pseudo-population)을 만드는 것입니다. 이 가상 모집단에서는 교란 요인($L$)과 치료($A$)의 연결고리가 끊어집니다.안정화 가중치 (Stabilized Weights) 공식논문의 식(14)에 해당하는 안정화 가중치는 다음과 같이 계산됩니다.
$$SW_i = \prod_{k=0}^{K} \frac{P(A_k | \bar{A}_{k-1})}{P(A_k | \bar{A}_{k-1}, \bar{L}_k)}$$
- 분모 (Denominator): 교란변수($L$)를 포함하여, 실제 받은 치료를 받을 확률 (성향 점수).
- 분자 (Numerator): 교란변수($L$)를 제외하고, 이전 치료 이력만으로 현재 치료를 받을 확률. (가중치의 분산을 줄여줌)

In [9]:
# IPTW 계산

#  분모 모델 (Denominator Model) 
# P(A_k | A_k-1, L_k): 교란변수 L을 포함하여 치료받을 확률 추정
denom_model = smf.glm('A ~ A_prev + L + time', 
                      data=df, 
                      family=sm.families.Binomial()).fit()

# 예측 확률 계산
df['prob_A_denom'] = denom_model.predict(df)

#  분자 모델 (Numerator Model) 
# P(A_k | A_k-1): 교란변수 L을 제외하고 치료받을 확률 추정 (Stabilization용)
num_model = smf.glm('A ~ A_prev + time', 
                    data=df, 
                    family=sm.families.Binomial()).fit()

df['prob_A_num'] = num_model.predict(df)

#  실제 관측된 치료(A)에 대한 확률 계산
# A=1이면 p, A=0이면 1-p 사용
df['prob_den_actual'] = np.where(df['A'] == 1, df['prob_A_denom'], 1 - df['prob_A_denom'])
df['prob_num_actual'] = np.where(df['A'] == 1, df['prob_A_num'], 1 - df['prob_A_num'])

# 시점별 가중치 계산
df['weight_t'] = df['prob_num_actual'] / df['prob_den_actual']

# 2-4. 개인별 누적 가중치 계산 (Product over time) - Eq 17
# 각 개인(id)별로 모든 시점의 가중치를 곱함
df['iptw'] = df.groupby('id')['weight_t'].transform('prod')


## 2. MSM (Marginal Structural Model) 모형 적합 및 비교 (Model Fitting)
이제 계산된 가중치를 사용하여 인과 효과를 추정합니다. 데이터를 개인 단위(Person-level)로 요약한 후 분석합니다


- **Naive Model (Unweighted):** 가중치 없이 일반 로지스틱 회귀를 사용하여 관측된 데이터의 연관성을 모델링합니다.
    $$\text{logit } P(Y=1 | \bar{A}=\bar{a}) = \beta'_0 + \beta'_1 \text{cum}(\bar{a})$$
    여기서 $\text{cum}(\bar{a})$는 누적 치료량을 의미합니다. 이 모델의 계수 $\beta'_1$은 교란 변수 $L$에 의해 편향되어 있어 인과 효과 $\beta_1$과 다릅니다 ($\beta_1 \neq \beta'_1$). 아픈 사람이 치료를 더 많이 받는 경향 때문에, 치료 효과가 과소평가되거나 오히려 해로운 것처럼 보일 수 있습니다.

- **MSM (Weighted):** IPTW 가중치($SW_i$)를 적용하여 로지스틱 회귀를 돌립니다.이는 반사실적(Counterfactual) 결과에 대한 인과 모형을 추정하는 것입니다.
    $$\text{logit } P(Y_{\bar{a}}=1) = \beta_0 + \beta_1 \text{cum}(\bar{a})$$
  이때 각 개인에게 가중치 $SW_i$를 부여하여 가상 모집단(Pseudo-population)을 생성하면, 치료 내역 $\bar{A}$와 교란 변수 $\bar{L}$ 사이의 연결 고리가 끊어지게 됩니다. 따라서 가중치가 적용된 회귀분석의 추정치는 인과 파라미터 $\beta_1$의 비편향 추정량(Unbiased Estimator)이 됩니다.

In [12]:

# 분석을 위해 데이터 요약 (개인별 1행으로 만듦)
# 누적 치료량(Cumulative Dose) 계산
df_person = df.groupby('id').agg({
    'Y': 'first',          # 결과 변수
    'A': 'sum',            # 누적 치료량 (Cumulative Exposure)
    'iptw': 'first'        # 계산된 가중치
}).rename(columns={'A': 'cum_A'})

print("\n--- Summary of Weights ---")
print(df_person['iptw'].describe())

#  비교: 가중치를 적용하지 않은 일반 로지스틱 회귀 (Biased)
# Eq 13 (Association Model)
naive_model = smf.glm('Y ~ cum_A', 
                      data=df_person, 
                      family=sm.families.Binomial()).fit()

#  MSM: IPTW 가중치를 적용한 로지스틱 회귀 (Causal)
# Eq 12 (Causal Model)
msm_model = smf.glm('Y ~ cum_A', 
                    data=df_person, 
                    freq_weights=df_person['iptw'], # 가중치 적용
                    family=sm.families.Binomial()).fit()



--- Summary of Weights ---
count    2000.000000
mean        0.997203
std         0.766158
min         0.207137
25%         0.479376
50%         0.788628
75%         1.269849
max         6.938674
Name: iptw, dtype: float64


## 3. 결과 확인 및 비교

In [14]:

print("\n" + "="*50)
print("Result Comparison")
print("="*50)

print("\n1. Naive Association Model (Standard Logistic Regression)")
print("   - Confounded estimate (likely biased)")
print(naive_model.summary().tables[1])

print("\n2. Marginal Structural Model (IPTW Weighted)")
print("   - Causal estimate (adjusting for time-dependent confounding)")

# 논문 섹션 6.3에 따라 Robust Standard Error (HC0 or Cluster) 사용 
# statsmodels의 get_margeff나 cov_type을 사용하여 강건 표준오차 확인 가능
print(msm_model.summary().tables[1])



Result Comparison

1. Naive Association Model (Standard Logistic Regression)
   - Confounded estimate (likely biased)
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.1682      0.150      7.791      0.000       0.874       1.462
cum_A         -0.8792      0.055    -15.871      0.000      -0.988      -0.771

2. Marginal Structural Model (IPTW Weighted)
   - Causal estimate (adjusting for time-dependent confounding)
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.2659      0.148      8.531      0.000       0.975       1.557
cum_A         -0.9015      0.055    -16.510      0.000      -1.009      -0.794
